# Sleep Duration and Health Indicators Analysis — NHANES 2017–2018

This portfolio notebook presents the complete analytical workflow used to study associations between sleep duration, demographic characteristics, health indicators, and lifestyle factors among 4,103 U.S. adults.

**Workflow:** data integration → cleaning and validation → feature engineering → exploratory analysis → Chi-square tests → Cramer's V effect sizes → interpretation.

> NHANES is cross-sectional. Results describe associations and must not be interpreted as causal relationships.


## 1. Load and prepare the NHANES modules

Each source table is cleaned independently before integration. Variable codes are renamed, relevant fields are selected, missing-value codes are converted to null values, and categorical responses are decoded.


In [ ]:
import pandas as pd
import numpy as np

demo=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\DEMO_J.xpt")


demo=demo.rename(columns={'RIAGENDR':'Gender',
                                'RIDAGEYR':'Age',
                                'RIDRETH3':'Race',
                                'DMDEDUC2':'Education',
                                'DMDMARTL':'Marital_Status',
                                'INDFMPIR':'Poverty_Index'}) #Annual family income

df_demo=demo[['SEQN','Gender','Age','Race','Education','Marital_Status','Poverty_Index']].copy()



categorical_columns = [
    "Education",
    "Marital_Status"]

df_demo[categorical_columns] = df_demo[categorical_columns].replace({
    7: np.nan,
    9: np.nan,
    77: np.nan,
    99: np.nan
})



df_demo["Age"] = (
    df_demo["Age"]
    .replace(5.397605346934028e-79, 0)
)

df_demo["Poverty_Index"] = (
    df_demo["Poverty_Index"]
    .replace(5.397605346934028e-79, 0)
)


df_demo['Gender']=df_demo['Gender'].replace({1:'Male',2:'Female'})

df_demo["Race"] = df_demo["Race"].replace({
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multi-Racial"
})

df_demo["Education"] = df_demo["Education"].replace({
    1: "Less than 9th grade",
    2: "9-11th grade",
    3: "High school/GED",
    4: "Some college/AA degree",
    5: "College graduate or above"
})

df_demo["Marital_Status"] = df_demo["Marital_Status"].replace({
    1: "Married",
    2: "Widowed",
    3: "Divorced",
    4: "Separated",
    5: "Never married",
    6: "Living with partner"
})


df_demo.head()


In [ ]:
sleep=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\SLQ_J.xpt")

sleep = sleep.rename(columns={
    'SLQ300': 'Bedtime_Workdays',
    'SLQ310': 'Wakeup_Workdays',
    'SLD012': 'Sleep_Hours_Workdays',

    'SLQ320': 'Bedtime_Weekend',
    'SLQ330': 'Wakeup_Weekend',
    'SLD013': 'Sleep_Hours_Weekend',

    'SLQ030': 'Snoring_Frequency',
    'SLQ040': 'Breathing_Interruptions_Frequency',
    'SLQ050': 'Reported_Sleep_Trouble_To_Doctor',
    'SLQ120': 'Daytime_Sleepiness_Frequency'
})


df_sleep = sleep[[
    'SEQN',

    'Bedtime_Workdays',
    'Wakeup_Workdays',
    'Sleep_Hours_Workdays',

    'Bedtime_Weekend',
    'Wakeup_Weekend',
    'Sleep_Hours_Weekend',

    'Snoring_Frequency',
    'Breathing_Interruptions_Frequency',
    'Reported_Sleep_Trouble_To_Doctor',
    'Daytime_Sleepiness_Frequency'
]].copy()


categorical_columns = [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Reported_Sleep_Trouble_To_Doctor",
    "Daytime_Sleepiness_Frequency"
]

df_sleep[categorical_columns] = df_sleep[categorical_columns].replace({
    7: np.nan,
    9: np.nan
})


time_columns = [
    "Bedtime_Workdays",
    "Wakeup_Workdays",
    "Bedtime_Weekend",
    "Wakeup_Weekend"]

for col in time_columns: #Transformamos el tipo de objeto bytes de las columnas de tiempo #We convert the byte object type of the time columns
    df_sleep[col] = df_sleep[col].str.decode("utf-8")
    

for col in [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Daytime_Sleepiness_Frequency"
]:
    df_sleep[col] = df_sleep[col].replace(5.397605346934028e-79, 0)    
    

    

frequency_labels_sleep = {
    0: "Never",
    1: "Rarely (1-2 nights/week)",
    2: "Occasionally (3-4 nights/week)",
    3: "Frequently (5+ nights/week)"
}

for col in [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency"
]:
    df_sleep[col] = df_sleep[col].replace(frequency_labels_sleep)

daytime_sleepiness_labels = {
    0: "Never",
    1: "Rarely (1 time/month)",
    2: "Sometimes (2-4 times/month)",
    3: "Often (5-15 times/month)",
    4: "Almost always (16-30 times/month)"
}

df_sleep['Daytime_Sleepiness_Frequency'] = df_sleep['Daytime_Sleepiness_Frequency'].replace(daytime_sleepiness_labels)

df_sleep["Reported_Sleep_Trouble_To_Doctor"] = (
    df_sleep["Reported_Sleep_Trouble_To_Doctor"]
    .replace({
        1: "Yes",
        2: "No"
    })
)


df_sleep.head()



In [ ]:


body=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\BMX_J.xpt")


body = body.rename(columns={
    "BMXWT": "Weight",
    "BMXHT": "Height",
    "BMXBMI": "BMI",
    "BMXWAIST": "Waist_Circumference",
    "BMXHIP": "Hip_Circumference"
})


df_body = body[[
    'SEQN',
    "Weight",
    "Height",
    "BMI",
    "Waist_Circumference",
    "Hip_Circumference"
]].copy()




df_body.head()


In [ ]:


df_ghb=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\GHB_J.xpt")


df_ghb = df_ghb.rename(columns={
    'LBXGH': 'HbA1c' #Es una medida que indica el nivel medio de glucosa en sangre durante los últimos 2-3 meses. 
})

df_ghb = df_ghb[['SEQN', 'HbA1c']].copy()



df_ghb
    

In [ ]:

pressure=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\BPX_J.xpt")

for col in ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]:
    pressure[col] = pressure[col].replace(5.397605346934028e-79, 0)

pressure = pressure.rename(columns={
    'BPXSY1': 'Systolic_BP_1',
    'BPXSY2': 'Systolic_BP_2',
    'BPXSY3': 'Systolic_BP_3',
    'BPXSY4': 'Systolic_BP_4',

    'BPXDI1': 'Diastolic_BP_1',
    'BPXDI2': 'Diastolic_BP_2',
    'BPXDI3': 'Diastolic_BP_3',
    'BPXDI4': 'Diastolic_BP_4'
})

df_pressure = pressure[[
    'SEQN',
    'Systolic_BP_1',
    'Systolic_BP_2',
    'Systolic_BP_3',
    'Systolic_BP_4',
    'Diastolic_BP_1',
    'Diastolic_BP_2',
    'Diastolic_BP_3',
    'Diastolic_BP_4'
]].copy()





df_pressure["Systolic_BP"] = df_pressure[
    ["Systolic_BP_1","Systolic_BP_2","Systolic_BP_3","Systolic_BP_4"]
].mean(axis=1)

df_pressure["Diastolic_BP"] = df_pressure[
    ["Diastolic_BP_1","Diastolic_BP_2","Diastolic_BP_3","Diastolic_BP_4"]
].mean(axis=1)



df_pressure=df_pressure[['SEQN','Systolic_BP','Diastolic_BP']].copy()


df_pressure.head()

In [ ]:

df_hdl=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\HDL_J.xpt")


df_hdl = df_hdl.rename(columns={
    'LBDHDD': 'HDL' #HDL significa High-Density Lipoprotein, conocido como colesterol HDL o "colesterol bueno".
})

df_hdl = df_hdl[['SEQN', 'HDL']].copy()

df_hdl.head()

In [ ]:
import pandas as pd


df_paq=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\PAQ_J.xpt")

df_paq = df_paq.rename(columns={
    'PAQ605': 'Vigorous_Work',
    'PAQ610': 'Vigorous_Work_Days',
    'PAD615': 'Vigorous_Work_Minutes',

    'PAQ620': 'Moderate_Work',
    'PAQ625': 'Moderate_Work_Days',
    'PAD630': 'Moderate_Work_Minutes',

    'PAQ635': 'Active_Transport',
    'PAQ640': 'Active_Transport_Days',
    'PAD645': 'Active_Transport_Minutes',

    'PAQ650': 'Vigorous_Recreation',
    'PAQ655': 'Vigorous_Recreation_Days',
    'PAD660': 'Vigorous_Recreation_Minutes',

    'PAQ665': 'Moderate_Recreation',
    'PAQ670': 'Moderate_Recreation_Days',
    'PAD675': 'Moderate_Recreation_Minutes',

    'PAD680': 'Sedentary_Minutes_Per_Day'
})


df_paq = df_paq[[
    "SEQN",

    "Vigorous_Work",
    "Vigorous_Work_Days",
    "Vigorous_Work_Minutes",

    "Moderate_Work",
    "Moderate_Work_Days",
    "Moderate_Work_Minutes",

    "Active_Transport",
    "Active_Transport_Days",
    "Active_Transport_Minutes",

    "Vigorous_Recreation",
    "Vigorous_Recreation_Days",
    "Vigorous_Recreation_Minutes",

    "Moderate_Recreation",
    "Moderate_Recreation_Days",
    "Moderate_Recreation_Minutes",

    "Sedentary_Minutes_Per_Day"
]].copy()



categorical_columns = [
    "Vigorous_Work",
    "Moderate_Work",
    "Active_Transport",
    "Vigorous_Recreation",
    "Moderate_Recreation"
]

days_columns = [
    "Vigorous_Work_Days",
    "Moderate_Work_Days",
    "Active_Transport_Days",
    "Vigorous_Recreation_Days",
    "Moderate_Recreation_Days"
]

minutes_columns = [
    "Vigorous_Work_Minutes",
    "Moderate_Work_Minutes",
    "Active_Transport_Minutes",
    "Vigorous_Recreation_Minutes",
    "Moderate_Recreation_Minutes",
    "Sedentary_Minutes_Per_Day"
]

df_paq[categorical_columns] = df_paq[categorical_columns].replace(9, np.nan)
df_paq[days_columns] = df_paq[days_columns].replace(99, np.nan)
df_paq[minutes_columns] = df_paq[minutes_columns].replace(9999, np.nan)

df_paq["Sedentary_Minutes_Per_Day"] = (df_paq["Sedentary_Minutes_Per_Day"].replace(5.397605346934028e-79, 0))


activity_labels = {
    1: "Yes",
    2: "No"
}

df_paq[categorical_columns] = (
    df_paq[categorical_columns]
    .replace(activity_labels)
)

df_paq.head()


In [ ]:
import pandas as pd
import numpy as np


smq=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\SMQ_J.xpt")


smq = smq.rename(columns={
    'SMQ020': 'Ever_Smoked_100_Cigarettes',
    'SMD030': 'Smoking_Start_Age',
    'SMQ040': 'Current_Smoking_Status',
    'SMQ050Q': 'Time_Since_Quitting',
    'SMQ050U': 'Time_Since_Quitting_Unit',
    'SMD057': 'Cigarettes_Per_Day_When_Quit'
})

df_smoking = smq[[
    'SEQN',
    'Ever_Smoked_100_Cigarettes',
    'Smoking_Start_Age',
    'Current_Smoking_Status',
    'Time_Since_Quitting',
    'Time_Since_Quitting_Unit'
]].copy()

categorical_columns = [
    "Ever_Smoked_100_Cigarettes",
    "Current_Smoking_Status",
    "Time_Since_Quitting_Unit"
]

df_smoking[categorical_columns] = df_smoking[categorical_columns].replace({
    7: np.nan,
    9: np.nan
})

df_smoking["Smoking_Start_Age"] = (
    df_smoking["Smoking_Start_Age"]
    .replace(5.397605346934028e-79, 0)
)

numeric_columns = [
    "Smoking_Start_Age",
    "Time_Since_Quitting"
]

df_smoking[numeric_columns] = df_smoking[numeric_columns].replace({
    777: np.nan,
    999: np.nan
})



df_smoking["Ever_Smoked_100_Cigarettes"] = (
    df_smoking["Ever_Smoked_100_Cigarettes"]
    .replace({
        1: "Yes",
        2: "No"
    })
)

df_smoking["Current_Smoking_Status"] = (
    df_smoking["Current_Smoking_Status"]
    .replace({
        1: "Every day",
        2: "Some days",
        3: "Not at all"
    })
)

df_smoking["Time_Since_Quitting_Unit"] = (
    df_smoking["Time_Since_Quitting_Unit"]
    .replace({
        1: "Days",
        2: "Weeks",
        3: "Months",
        4: "Years"
    })
)

df_smoking.head()

In [ ]:
import pandas as pd
import numpy as np

df_alcohol=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\ALQ_J.xpt")


df_alcohol = df_alcohol.rename(columns={
    "ALQ111": "Ever_Had_Alcohol",
    "ALQ121": "Alcohol_Frequency_Last_12_Months",
    "ALQ130": "Average_Drinks_Per_Day",
    "ALQ142": "Heavy_Drinking_Frequency_Last_12_Months",
    "ALQ270": "Binge_Drinking_2h_Frequency_Last_12_Months",
    "ALQ280": "Days_With_8_Or_More_Drinks_Last_12_Months",
    "ALQ290": "Days_With_12_Or_More_Drinks_Last_12_Months",
    "ALQ151": "Ever_Been_Heavy_Drinker",
    "ALQ170": "Binge_Drinking_Last_30_Days"
})

df_alcohol = df_alcohol[[
    "SEQN",
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months",
    "Average_Drinks_Per_Day", # Value 15 represents "15 or more drinks"
    "Binge_Drinking_Last_30_Days"
]].copy()


categorical_columns = [
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months"
]

df_alcohol[categorical_columns] = df_alcohol[categorical_columns].replace({
    7: np.nan,
    9: np.nan,
    77: np.nan,
    99: np.nan
})

numeric_columns = [
    "Average_Drinks_Per_Day",
    "Binge_Drinking_Last_30_Days"
]

df_alcohol[numeric_columns] = df_alcohol[numeric_columns].replace({
    777: np.nan,
    999: np.nan
})



columns_with_zero_issue = [
    "Alcohol_Frequency_Last_12_Months",
    "Binge_Drinking_Last_30_Days"
]

for col in columns_with_zero_issue:
    df_alcohol[col] = df_alcohol[col].replace(
        5.397605346934028e-79,
        0
    )


df_alcohol["Ever_Had_Alcohol"] = (
    df_alcohol["Ever_Had_Alcohol"]
    .replace({
        1: "Yes",
        2: "No"
    })
)

alcohol_frequency_labels = {
    0: "Never in the last 12 months",
    1: "Every day",
    2: "Nearly every day",
    3: "3-4 times per week",
    4: "Twice a week",
    5: "Once a week",
    6: "2-3 times per month",
    8: "Once a month",
    10: "3-11 times in the last year"
}

df_alcohol["Alcohol_Frequency_Last_12_Months"] = (
    df_alcohol["Alcohol_Frequency_Last_12_Months"]
    .replace(alcohol_frequency_labels)
)

df_alcohol.head()



## 2. Integrate and validate the analytical dataset

The demographic module is used as the base table and the remaining modules are joined through the unique participant identifier (`SEQN`). Duplicate identifiers, dimensions, and missingness are checked after integration.


In [ ]:
dfs = {
    'demo': df_demo,
    'sleep': df_sleep,
    'smoking': df_smoking,
    'alcohol': df_alcohol,
    'body': df_body,
    'physical_activity': df_paq,
    'Glycohemoglobin': df_ghb,
    'Blood Pressure': df_pressure,
    'Cholesterol': df_hdl
}

df_final = dfs['demo'].copy()

for dataset_name, df_current in dfs.items():

    if dataset_name != 'demo':

        df_final = df_final.merge(
            df_current,
            on='SEQN',
            how='left'
        )

        print(f'{dataset_name}: {df_final.shape}')

In [ ]:
for dataset_name, current_df in dfs.items():

    duplicated_ids = current_df['SEQN'].duplicated().sum()

    print(dataset_name, duplicated_ids)

In [ ]:
print("=" * 40)
print("DATAFRAME")
print("=" * 40)

print(f"Rows: {df_final.shape[0]}")
print(f"Columns: {df_final.shape[1]}")

print(f"\nDuplicated SEQN values: {df_final['SEQN'].duplicated().sum()}")


In [ ]:
na = pd.DataFrame({
    "NaN": df_final.isna().sum(),
    "%": (df_final.isna().mean() * 100).round(2),
    "Data Type": df_final.dtypes
})

na = (
    na[na["NaN"] > 0]
      .sort_values("%", ascending=False)
)

na

## 3. Feature engineering

This section creates interpretable analytical variables for smoking, alcohol consumption, BMI, physical activity, average weekly sleep duration, and sleep-duration categories.


In [ ]:
df_analysis = df_final.copy()

print(f"Participants before filtering: {len(df_analysis)}")

df_analysis = df_analysis[df_analysis["Age"] >= 20].copy()

print(f"Participants after filtering: {len(df_analysis)}")

df_analysis.reset_index(drop=True, inplace=True)

In [ ]:
df_analysis["Smoking_Status"] = pd.Series(dtype="object")

df_analysis.loc[
    df_analysis["Ever_Smoked_100_Cigarettes"] == "No",
    "Smoking_Status"
] = "Never smoker"

df_analysis.loc[
    df_analysis["Current_Smoking_Status"] == "Not at all",
    "Smoking_Status"
] = "Former smoker"

df_analysis.loc[
    df_analysis["Current_Smoking_Status"].isin(["Every day", "Some days"]),
    "Smoking_Status"
] = "Current smoker"


df_analysis.drop(
    columns=[
        "Ever_Smoked_100_Cigarettes",
        "Current_Smoking_Status",
        "Smoking_Start_Age",
        "Time_Since_Quitting",
        "Time_Since_Quitting_Unit"
    ],
    inplace=True,
    errors="ignore"
)


In [ ]:
df_analysis["Smoking_Status"].value_counts(dropna=False)

round(
    df_analysis["Smoking_Status"]
        .value_counts(dropna=False, normalize=True) * 100,
    2
)

In [ ]:

df_analysis["Alcohol_Consumption"] = pd.Series(dtype="object")

df_analysis.loc[
    df_analysis["Ever_Had_Alcohol"] == "No",
    "Alcohol_Consumption"
] = "Never"

df_analysis.loc[
    (df_analysis["Ever_Had_Alcohol"] == "Yes") &
    (df_analysis["Alcohol_Frequency_Last_12_Months"] == "Never in the last 12 months"),
    "Alcohol_Consumption"
] = "Former"

df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"] == "3-11 times in the last year",
    "Alcohol_Consumption"
] = "Rare"


df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Once a month",
        "2-3 times per month"
    ]),
    "Alcohol_Consumption"
] = "Occasional"

df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Once a week",
        "Twice a week"
    ]),
    "Alcohol_Consumption"
] = "Regular"

df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"] == "3-4 times per week",
    "Alcohol_Consumption"
] = "Frequent"

df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Nearly every day",
        "Every day"
    ]),
    "Alcohol_Consumption"
] = "Daily"


order = [
    "Never",
    "Former",
    "Rare",
    "Occasional",
    "Regular",
    "Frequent",
    "Daily"
]

df_analysis["Alcohol_Consumption"] = pd.Categorical(
    df_analysis["Alcohol_Consumption"],
    categories=order,
    ordered=True
)


df_alcohol.head()

In [ ]:
pd.crosstab(
    df_analysis["Ever_Had_Alcohol"],
    df_analysis["Alcohol_Consumption"],
    dropna=False
)

In [ ]:

df_analysis["BMI_Category"] = pd.Series(dtype="object")

df_analysis.loc[
    df_analysis["BMI"] < 18.5,
    "BMI_Category"
] = "Underweight"

df_analysis.loc[
    (df_analysis["BMI"] >= 18.5) &
    (df_analysis["BMI"] < 25),
    "BMI_Category"
] = "Normal"

df_analysis.loc[
    (df_analysis["BMI"] >= 25) &
    (df_analysis["BMI"] < 30),
    "BMI_Category"
] = "Overweight"

df_analysis.loc[
    df_analysis["BMI"] >= 30,
    "BMI_Category"
] = "Obese"



bmi_order = [
    "Underweight",
    "Normal",
    "Overweight",
    "Obese"
]

df_analysis["BMI_Category"] = pd.Categorical(
    df_analysis["BMI_Category"],
    categories=bmi_order,
    ordered=True
)



In [ ]:

moderate_recreation = (
    df_analysis["Moderate_Recreation_Days"] *
    df_analysis["Moderate_Recreation_Minutes"]
)

vigorous_recreation = (
    df_analysis["Vigorous_Recreation_Days"] *
    df_analysis["Vigorous_Recreation_Minutes"]
)

active_transport = (
    df_analysis["Active_Transport_Days"] *
    df_analysis["Active_Transport_Minutes"]
)

df_analysis["Physical_Activity_Equivalent_Minutes_Week"] = (
    moderate_recreation.fillna(0)
    + active_transport.fillna(0)
    + 2 * vigorous_recreation.fillna(0)
)


In [ ]:

df_analysis["Physical_Activity_Level"] = pd.cut(
    df_analysis["Physical_Activity_Equivalent_Minutes_Week"],
    bins=[-1, 0, 149, 299, float("inf")],
    labels=[
        "Inactive",
        "Below Recommendation",
        "Meets Recommendation",
        "Exceeds Recommendation"
    ]
)

activity_order = [
    "Inactive",
    "Below Recommendation",
    "Meets Recommendation",
    "Exceeds Recommendation"
]

df_analysis["Physical_Activity_Level"] = pd.Categorical(
    df_analysis["Physical_Activity_Level"],
    categories=activity_order,
    ordered=True
)

display(df_analysis["Physical_Activity_Level"].value_counts(dropna=False))

In [ ]:


df_analysis["Average_Sleep_Hours"] = (
    (
        df_analysis["Sleep_Hours_Workdays"] * 5
        + df_analysis["Sleep_Hours_Weekend"] * 2
    ) / 7
)


In [ ]:

df_analysis["Sleep_Duration_Category"] = pd.Series(dtype='object')

df_analysis.loc[
    df_analysis["Average_Sleep_Hours"] < 7,
    "Sleep_Duration_Category"
] = "Short Sleep"

df_analysis.loc[
    (df_analysis["Average_Sleep_Hours"] >= 7) &
    (df_analysis["Average_Sleep_Hours"] <= 9),
    "Sleep_Duration_Category"
] = "Recommended Sleep"

df_analysis.loc[
    df_analysis["Average_Sleep_Hours"] > 9,
    "Sleep_Duration_Category"
] = "Long Sleep"

sleep_order = [
    "Short Sleep",
    "Recommended Sleep",
    "Long Sleep"
]

df_analysis["Sleep_Duration_Category"] = pd.Categorical(
    df_analysis["Sleep_Duration_Category"],
    categories=sleep_order,
    ordered=True
)


In [ ]:

columns_to_drop = [
    "Moderate_Work_Days",
    "Moderate_Work_Minutes",
    "Vigorous_Work_Days",
    "Vigorous_Work_Minutes",
    "Moderate_Recreation_Days",
    "Moderate_Recreation_Minutes",
    "Vigorous_Recreation_Days",
    "Vigorous_Recreation_Minutes",
    "Active_Transport_Days",
    "Active_Transport_Minutes"
]

df_analysis.drop(columns=columns_to_drop, inplace=True,errors='ignore')
print(f"{len(columns_to_drop)} original alcohol variables were removed.")



In [ ]:

physical_activity_binary_columns = [
    "Vigorous_Work",
    "Moderate_Work",
    "Active_Transport",
    "Vigorous_Recreation",
    "Moderate_Recreation"
    
]

df_analysis.drop(columns=physical_activity_binary_columns, inplace=True,errors='ignore')

print(f"{len(physical_activity_binary_columns)} binary physical activity variables were removed.")


In [ ]:

alcohol_columns = [
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months",
    "Average_Drinks_Per_Day",
    "Binge_Drinking_Last_30_Days"
]

df_analysis.drop(columns=alcohol_columns, inplace=True,errors='ignore')

print(f"{len(alcohol_columns)} original alcohol variables were removed.")

In [ ]:

sleep_columns = [
    "Bedtime_Workdays",
    "Wakeup_Workdays",
    "Bedtime_Weekend",
    "Wakeup_Weekend"
]

df_analysis.drop(columns=sleep_columns, inplace=True,errors='ignore')

print(f"{len(sleep_columns)} original sleep schedule variables were removed.")

## 4. Final analytical dataset

Redundant source variables are removed after validation while variables required for the research questions are retained.


In [ ]:

body_measurement_columns = [
    "Weight",
    "Height",
    "Hip_Circumference"
]

df_analysis.drop(columns=body_measurement_columns, inplace=True,errors='ignore')

print(f"{len(body_measurement_columns)} body measurement variables were removed.")


## 5. Exploratory data analysis

The study population and the distributions of demographic, sleep, health, and lifestyle variables are examined before statistical testing.


In [ ]:

df_analysis["Age_Group"] = pd.cut(
    df_analysis["Age"],
    bins=[20, 40, 60, 80],
    labels=["20-39", "40-59", "60-80"],
    include_lowest=True)

group_median = (df_analysis.groupby(["Gender", "Education", "Age_Group"],observed=False)["Poverty_Index"].transform("median"))

df_analysis.drop(columns="Age_Group", inplace=True)

df_analysis["Poverty_Index"] = (
    df_analysis["Poverty_Index"]
    .fillna(group_median))

df = df_analysis.dropna(
    subset=["Alcohol_Consumption"]
).copy()

print("Original:", df_analysis.shape)
print("Sin NaN en Alcohol_Consumption:", df.shape)

df.isna().sum().sort_values(ascending=False)

df = df.reset_index(drop=True)


In [ ]:



numerical_variables = ["Age", "Poverty_Index"]

summary_statistics = df[numerical_variables].describe().T

summary_statistics["Missing"] = (
    df[numerical_variables]
    .isna()
    .sum()
)

summary_statistics = summary_statistics[
    ["count", "Missing", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(summary_statistics.round(2))

In [ ]:
study_population_cat = df[[
    "Gender",
    "Race",
    "Education",
    "Marital_Status"
]]

for col in study_population_cat:
    print(f"\n{'='*60}")
    print(col)
    print('='*60)

    freq = (
        df[col]
        .value_counts(dropna=False)
        .rename("Count")
        .to_frame()
    )

    freq["Percentage"] = (
        freq["Count"] / len(df) * 100
    ).round(2)

    display(freq)
   

In [ ]:
sleep_stats = df["Average_Sleep_Hours"].describe().to_frame()

display(sleep_stats)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="Average_Sleep_Hours",
    bins=20,
    kde=True
)

plt.title("Distribution of Average Sleep Duration")
plt.xlabel("Average Sleep Hours")
plt.ylabel("Frequency")

plt.show()
print('Asimetría: ',df["Average_Sleep_Hours"].skew())

In [ ]:
plt.figure(figsize=(8,2.5))

sns.boxplot(
    x=df["Average_Sleep_Hours"]
)

plt.title("Boxplot of Average Sleep Duration")

plt.show()

In [ ]:
Q1 = df["Average_Sleep_Hours"].quantile(0.25)
Q3 = df["Average_Sleep_Hours"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["Average_Sleep_Hours"] < lower) |
    (df["Average_Sleep_Hours"] > upper)
]

print("Lower limit:", lower)
print("Upper limit:", upper)
print("Number of outliers:", len(outliers))

In [ ]:

sleep_category = (
    df["Sleep_Duration_Category"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

sleep_category["Percentage"] = (
    sleep_category["Count"] / len(df) * 100
).round(2)

print(sleep_category)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

order = [
    "Short Sleep",
    "Recommended Sleep",
    "Long Sleep"
]

plt.figure(figsize=(8, 5))

ax = sns.countplot(
    data=df,
    x="Sleep_Duration_Category",
    order=order,
    color="steelblue"
)

for container in ax.containers:
    ax.bar_label(container, fontsize=10)

plt.title("Distribution of Sleep Duration Categories", fontsize=14)
plt.xlabel('')
plt.ylabel("Number of Participants", fontsize=12)

plt.tight_layout()
plt.show()

print(df.groupby('Sleep_Duration_Category').size())


In [ ]:

BMI_Points = np.select([df["BMI_Category"] == "Normal", df["BMI_Category"].isin(["Underweight", "Overweight"])
                        ,df["BMI_Category"] == "Obese"], [2,1,0], default=np.nan)

HbA1c_Points = np.select(
    [
        df["HbA1c"] < 5.7,
        (df["HbA1c"] >= 5.7) & (df["HbA1c"] < 6.5),
        df["HbA1c"] >= 6.5
    ],
    [2, 1, 0],
    default=np.nan
)

BP_Points = np.select(
    [
        (df["Systolic_BP"] < 120) &
        (df["Diastolic_BP"] < 80),

        (df["Systolic_BP"].between(120, 129)) &
        (df["Diastolic_BP"] < 80),

        (df["Systolic_BP"] >= 130) |
        (df["Diastolic_BP"] >= 80)
    ],
    [2, 1, 0],
    default=np.nan
)

HDL_Points = np.select(
    [
        ((df["Gender"] == "Male") & (df["HDL"] >= 40)) |
        ((df["Gender"] == "Female") & (df["HDL"] >= 50)),

        ((df["Gender"] == "Male") & (df["HDL"] < 40)) |
        ((df["Gender"] == "Female") & (df["HDL"] < 50))
    ],
    [2, 0],
    default=np.nan
)


In [ ]:


health_points = pd.DataFrame({
    "BMI": BMI_Points,
    "HbA1c": HbA1c_Points,
    "BP": BP_Points,
    "HDL": HDL_Points
})

obtained = health_points.sum(axis=1, skipna=True)

available = health_points.notna().sum(axis=1) * 2

df["Health_Profile_Percentage"] = (
    obtained / available * 100).round(2)

df.loc[available < 6, "Health_Profile_Percentage"] = np.nan


df["Health_Profile"] = np.select(
    [
        df["Health_Profile_Percentage"] >= 80,
        df["Health_Profile_Percentage"].between(50, 79.99),
        df["Health_Profile_Percentage"] < 50
    ],
    [
        "Good Health Profile",
        "Intermediate Health Profile",
        "Poor Health Profile"
    ],
    default=None
)

display(df["Health_Profile"].value_counts(dropna=False))

In [ ]:
Physical_Activity_Points = np.select(
    [
        df["Physical_Activity_Level"].isin(
            ["Meets Recommendation", "Exceeds Recommendation"]
        ),
        df["Physical_Activity_Level"] == "Below Recommendation",
        df["Physical_Activity_Level"] == "Inactive"
    ],
    [2, 1, 0],
    default=np.nan
)

In [ ]:
Smoking_Points = np.select(
    [
        df["Smoking_Status"] == "Never smoker",
        df["Smoking_Status"] == "Former smoker",
        df["Smoking_Status"] == "Current smoker"
    ],
    [2, 1, 0],
    default=np.nan
)

In [ ]:
Alcohol_Points = np.select(
    [
        df["Alcohol_Consumption"].isin(
            ["Never", "Rare", "Occasional"]
        ),
        df["Alcohol_Consumption"].isin(
            ["Former", "Regular"]
        ),
        df["Alcohol_Consumption"].isin(
            ["Frequent", "Daily"]
        )
    ],
    [2, 1, 0],
    default=np.nan
)

In [ ]:

lifestyle_points = pd.DataFrame({
    "Smoking": Smoking_Points,
    "Alcohol": Alcohol_Points,
    "Physical_Activity": Physical_Activity_Points
})

obtained = lifestyle_points.sum(axis=1, skipna=True)
available = lifestyle_points.notna().sum(axis=1) * 2

df["Lifestyle_Profile_Percentage"] = (
    obtained / available * 100
).round(2)

df.loc[
    available < 4,
    "Lifestyle_Profile_Percentage"
] = np.nan

df["Lifestyle_Profile"] = np.select(
    [
        df["Lifestyle_Profile_Percentage"] >= 80,
        df["Lifestyle_Profile_Percentage"].between(50, 79.9999),
        df["Lifestyle_Profile_Percentage"] < 50
    ],
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ],
    default=None
)



In [ ]:
df["Lifestyle_Profile"] = np.select(
    [
        df["Lifestyle_Profile_Percentage"] >= 80,
        df["Lifestyle_Profile_Percentage"].between(50, 79.9999),
        df["Lifestyle_Profile_Percentage"] < 50
    ],
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ],
    default=None
)

display(df["Lifestyle_Profile"].value_counts(dropna=False))

In [ ]:


import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

sns.boxplot(
    data=df,
    x="Gender",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Hours by Gender")
plt.xlabel("Gender")
plt.ylabel("Average Sleep Hours")

plt.show()

print(df.groupby("Gender")["Average_Sleep_Hours"].describe())

In [ ]:




age_bins = [20, 40, 60, float("inf")]
age_labels = [
    "Young Adults (20–39)",
    "Middle-aged Adults (40–59)",
    "Older Adults (60+)"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

print(df.groupby('Age_Group',observed=False)['Average_Sleep_Hours'].describe())


plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="Age_Group",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Average Sleep Duration (hours)")

plt.show()

In [ ]:


plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="BMI_Category",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across BMI Categories")
plt.xlabel("")
plt.ylabel("Average Sleep Duration (hours)")


plt.show()
print(df.groupby("BMI_Category",observed=False)["Average_Sleep_Hours"].describe())

## 6. Statistical association analysis

Contingency tables, percentage distributions, Chi-square tests of independence, and Cramer's V effect sizes are used to evaluate relationships with sleep-duration category.


In [ ]:




plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="Health_Profile",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across Health Profile Categories")
plt.xlabel("Health Profile")
plt.ylabel("Average Sleep Duration (hours)")

plt.show()

print(df.groupby('Health_Profile')['Average_Sleep_Hours'].describe())

In [ ]:



plt.figure(figsize=(8,6))
sns.boxplot(df,x='Lifestyle_Profile',y='Average_Sleep_Hours')

plt.xlabel("Lifestyle Profile")
plt.ylabel("Average Sleep Duration (hours)")
plt.title("Average Sleep Duration Across Lifestyle Profile Categories")
plt.show()

print(df.groupby("Lifestyle_Profile",observed=False)["Average_Sleep_Hours"].describe())

In [ ]:
health_sleep = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

display(health_sleep)

In [ ]:
health_sleep_pct = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"],
    normalize="index"
) * 100

display(health_sleep_pct.round(1))



In [ ]:
health_sleep_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(8,6)
)

plt.ylabel("Percentage")
plt.title("Health Profile Distribution Across Sleep Duration Categories")
plt.legend(title="Health Profile")
plt.xticks(rotation=0)
plt.xlabel('')
plt.show()

In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    kruskal,
    spearmanr,
    pearsonr
)


In [ ]:
target = "Sleep_Duration_Category"


categorical_vars = [

    "Gender",
    "Race",
    "Education",
    "Marital_Status",
    "Age_Group",

    "Smoking_Status",
    "Alcohol_Consumption",

    "BMI_Category",

    "Physical_Activity_Level",

    "Reported_Sleep_Trouble_To_Doctor",
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Daytime_Sleepiness_Frequency",

    "Health_Profile",
    "Lifestyle_Profile"

]


numeric_vars = [

    "Age",

    "Average_Sleep_Hours",

    "Sleep_Hours_Workdays",

    "Sleep_Hours_Weekend",

    "BMI",

    "Waist_Circumference",

    "Sedentary_Minutes_Per_Day",

    "Physical_Activity_Equivalent_Minutes_Week",

    "HbA1c",

    "HDL",

    "Systolic_BP",

    "Diastolic_BP",

    "Poverty_Index",

    "Health_Profile_Percentage",

    "Lifestyle_Profile_Percentage"

]

In [ ]:
def cramers_v(contingency):

    chi2 = chi2_contingency(contingency)[0]

    n = contingency.sum().sum()

    r, c = contingency.shape

    return np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

In [ ]:
categorical_results = []

for var in categorical_vars:

    temp = df[[target, var]].dropna()

    contingency = pd.crosstab(
        temp[target],
        temp[var]
    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    cv = cramers_v(contingency)

    categorical_results.append({

        "Variable": var,

        "Test": "Chi-square",

        "Chi2": round(chi2,2),

        "p_value": round(p,5),

        "Cramers_V": round(cv,3),

        "N": len(temp)

    })

categorical_results = pd.DataFrame(categorical_results)

categorical_results.sort_values(
    "p_value",
    inplace=True
)

display(categorical_results)

In [ ]:
numeric_results = []

for var in numeric_vars:

    temp = df[[target,var]].dropna()

    groups = [

        group[var].values

        for _, group in temp.groupby(target,observed=False)

    ]

    stat, p = kruskal(*groups)

    means = temp.groupby(target,observed=False)[var].mean()

    numeric_results.append({

        "Variable": var,

        "Test":"Kruskal",

        "Statistic": round(stat,2),

        "p_value": round(p,5),

        "Short_Mean": round(means.get("Short Sleep",np.nan),2),

        "Recommended_Mean": round(means.get("Recommended Sleep",np.nan),2),

        "Long_Mean": round(means.get("Long Sleep",np.nan),2),

        "N": len(temp)

    })

numeric_results = pd.DataFrame(numeric_results)

numeric_results.sort_values(
    "p_value",
    inplace=True
)

display(numeric_results)

In [ ]:
correlations = []

for i in range(len(numeric_vars)):

    for j in range(i+1,len(numeric_vars)):

        var1 = numeric_vars[i]

        var2 = numeric_vars[j]

        temp = df[[var1,var2]].dropna()

        corr,p = spearmanr(
            temp[var1],
            temp[var2]
        )

        correlations.append({

            "Variable_1":var1,

            "Variable_2":var2,

            "Correlation":round(corr,3),

            "p_value":round(p,5)

        })

correlations = pd.DataFrame(correlations)

correlations["abs_corr"] = correlations["Correlation"].abs()

correlations = correlations.sort_values(
    "abs_corr",
    ascending=False
)

display(correlations.head(20))

In [ ]:

all_categorical = categorical_vars + [target]

association_matrix = []

for i in range(len(all_categorical)):

    for j in range(i+1,len(all_categorical)):

        var1 = all_categorical[i]

        var2 = all_categorical[j]

        temp = df[[var1,var2]].dropna()

        contingency = pd.crosstab(
            temp[var1],
            temp[var2]
        )

        chi2,p,dof,expected = chi2_contingency(contingency)

        cv = cramers_v(contingency)

        association_matrix.append({

            "Variable_1":var1,

            "Variable_2":var2,

            "p_value":round(p,5),

            "Cramers_V":round(cv,3)

        })

association_matrix = pd.DataFrame(association_matrix)

association_matrix = association_matrix.sort_values(

    ["Cramers_V","p_value"],

    ascending=[False,True]

)

display(association_matrix.head(50))

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="Sleep_Duration_Category",
    y="BMI",
    order=["Short Sleep","Recommended Sleep","Long Sleep"]
)

plt.xlabel("Sleep Duration Category")
plt.ylabel("BMI")
plt.tight_layout()

plt.show()

In [ ]:

import matplotlib.pyplot as plt

health_percent = (
    pd.crosstab(
        df["Health_Profile"],
        df["Sleep_Duration_Category"],
        normalize="index"
    ) * 100
).round(1)



ax = health_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(12,5)
)

plt.ylabel("Percentage (%)")
plt.xlabel("Sleep Duration Category")
plt.xticks(rotation=0)
plt.legend(title="Health Profile", bbox_to_anchor=(1.02,1), loc="upper left")

for container in ax.containers:
    labels = [
        f"{bar.get_height():.1f}%"
        if bar.get_height() > 3 else ""
        for bar in container
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:

import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np


health_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Health_Profile"],
        normalize="index"
    ) * 100
).round(1)

display(health_percent)


ax = health_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(9,6)
)

plt.title("Health Profile Distribution by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)

plt.legend(
    title="Health Profile",
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)


for container in ax.containers:

    labels = [

        f"{bar.get_height():.1f}%"
        if bar.get_height() > 3
        else ""

        for bar in container

    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()


contingency = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency)


n = contingency.sum().sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print("=" * 40)
print("Statistical Results")
print("=" * 40)

print(f"Chi-square : {chi2:.2f}")
print(f"Degrees of freedom : {dof}")
print(f"P-value : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np


health_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Health_Profile"],
        normalize="index"
    ) * 100
).round(1)

print(health_percent)


colors = {
    "Good Health Profile": "#2ca02c",          # Verde
    "Intermediate Health Profile": "#ff7f0e",  # Naranja
    "Poor Health Profile": "#d62728"           # Rojo
}

fig, ax = plt.subplots(figsize=(10,6))

health_percent.plot(
    kind="bar",
    ax=ax,
    color=colors,
    width=0.8
)


ax.set_title(
    "Health Profile by Sleep Duration",
    fontsize=15,
    weight="bold"
)

ax.set_xlabel("Sleep Duration Category")
ax.set_ylabel("Percentage (%)")

ax.set_ylim(0,60)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.xticks(rotation=0)


for container in ax.containers:

    labels = [
        f"{bar.get_height():.1f}%"
        if bar.get_height() >= 3
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        padding=3,
        fontsize=9
    )


ax.legend(
    [
        "Good Health",
        "Intermediate Health",
        "Poor Health"
    ],
    title="Health Profile",
    frameon=False,
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


contingency = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency)


n = contingency.values.sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r-1, c-1))
)

print("="*40)
print("Statistical Results")
print("="*40)
print(f"Chi-square : {chi2:.2f}")
print(f"P-value    : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

In [ ]:
lifestyle_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Lifestyle_Profile"],
        normalize="index"
    ) * 100
).round(1)

lifestyle_percent = lifestyle_percent[
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ]
]

print(lifestyle_percent)



colors = [
     "#2ca02c",   # Verde Healthy Lifestyle
     "#ff7f0e",         # Naranja  recomend
     "#1f77b4"           # Azul   
]

fig, ax = plt.subplots(figsize=(9, 5.5))

lifestyle_percent.plot(
    kind="bar",
    stacked=True,
    color=colors,
    edgecolor="white",
    width=0.7,
    ax=ax
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%"
        if value >= 5 else ""
        for value in container.datavalues
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9,
        color="black"
    )

ax.set_title(
    "Lifestyle Profile Distribution by Sleep Duration Category",
    fontsize=14,
    weight="bold",
    pad=15
)

ax.set_xlabel("Sleep Duration Category", fontsize=11)
ax.set_ylabel("Participants (%)", fontsize=11)

ax.set_xticklabels([
    "Short",
    "Recommended",
    "Long"
], rotation=0, fontsize=10)

ax.tick_params(axis="y", labelsize=10)

ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", visible=False)

ax.legend(
    title="Lifestyle Profile",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=10,
    title_fontsize=11
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Lifestyle_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

lifestyle_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Lifestyle_Profile"],
        normalize="index"
    ) * 100
).round(1)

lifestyle_percent = lifestyle_percent[
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ]
]

heatmap_data = lifestyle_percent / 100

plt.figure(figsize=(7, 4.8))

ax = sns.heatmap(
    heatmap_data,
    annot=lifestyle_percent.astype(str) + "%",
    fmt="",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Proportion"}
)

ax.set_title(
    "Lifestyle Profile by Sleep Duration Category",
    fontsize=14,
    weight="bold"
)

ax.set_xlabel("Lifestyle Profile")
ax.set_ylabel("Sleep Duration Category")

ax.set_xticklabels(
    ["Healthy", "Intermediate", "Unhealthy"],
    rotation=0
)

ax.set_yticklabels(
    ["Short", "Recommended", "Long"],
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:
pa_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Physical_Activity_Level"],
        normalize="index"
    ) * 100
).round(1)

display(pa_percent)

colors = [
    "#d73027",  # Inactive
    "#fc8d59",  # Below Recommendation
    "#91cf60",  # Meets Recommendation
    "#1a9850"   # Exceeds Recommendation
]

ax = pa_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5),
    color=colors
)


for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

plt.title("Physical Activity Level by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)
plt.legend(
    title="Physical Activity Level",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Physical_Activity_Level"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
smoking_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Smoking_Status"],
        normalize="index"
    ) * 100
).round(1)

print(smoking_percent)

colors = [
    "#d73027",  # Current smoker (red)
    "#fc8d59",  # Former smoker (orange)
    "#1a9850"   # Never smoker (green)
]

ax = smoking_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=colors,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

plt.title("Smoking Status by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)
plt.legend(
    title="Smoking Status",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Smoking_Status"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
alcohol_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Alcohol_Consumption"],
        normalize="index"
    ) * 100
).round(1)

display(alcohol_percent)

colors = [
    "#1a9850",  # Never
    "#91cf60",  # Former
    "#fee08b",  # Rare
    "#fdae61",  # Occasional
    "#fc8d59",  # Regular
    "#d73027",  # Frequent
    "#a50026"   # Daily
]

ax = alcohol_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=colors,
    width=0.65,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

plt.title("Alcohol Consumption by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Alcohol Consumption",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Alcohol_Consumption"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
snoring_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Snoring_Frequency"],
        normalize="index"
    ) * 100
).round(1)

snoring_percent = snoring_percent[
    [
        "Never",
        "Rarely (1-2 nights/week)",
        "Occasionally (3-4 nights/week)",
        "Frequently (5+ nights/week)"
    ]
]

display(snoring_percent)

colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Occasionally
    "#d73027"   # Frequently
]

ax = snoring_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    width=0.7,
    color=colors,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

plt.title("Snoring Frequency by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Snoring Frequency",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Snoring_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
breathing_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Breathing_Interruptions_Frequency"],
        normalize="index"
    ) * 100
).round(1)

breathing_percent = breathing_percent[
    [
        "Never",
        "Rarely (1-2 nights/week)",
        "Occasionally (3-4 nights/week)",
        "Frequently (5+ nights/week)"
    ]
]

print(breathing_percent)

colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Occasionally
    "#d73027"   # Frequently
]

ax = breathing_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    width=0.7,
    color=colors,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 1 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

plt.title("Breathing Interruptions Frequency by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Breathing Interruptions Frequency",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Breathing_Interruptions_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
df
display(df.groupby('Marital_Status')['Average_Sleep_Hours'].mean().sort_values(ascending=False).round(2))

In [ ]:
sleep_trouble_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Reported_Sleep_Trouble_To_Doctor"],
        normalize="index"
    ) * 100
).round(1)

sleep_trouble_percent = sleep_trouble_percent[
    [
        "No",
        "Yes"
    ]
]

print(sleep_trouble_percent)

colors = [
    "#1a9850",  # No
    "#d73027"   # Yes
]

ax = sleep_trouble_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=colors,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

plt.title("Reported Sleep Trouble by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Reported Sleep Trouble",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Reported_Sleep_Trouble_To_Doctor"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
sleepiness_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Daytime_Sleepiness_Frequency"],
        normalize="index"
    ) * 100
).round(1)

sleepiness_percent = sleepiness_percent[
    [
        "Never",
        "Rarely (1 time/month)",
        "Sometimes (2-4 times/month)",
        "Often (5-15 times/month)",
        "Almost always (16-30 times/month)"
    ]
]

print(sleepiness_percent)

colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Sometimes
    "#fc8d59",  # Often
    "#d73027"   # Almost always
]

ax = sleepiness_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    color=colors,
    edgecolor="white"
)

for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

plt.title("Daytime Sleepiness Frequency by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Daytime Sleepiness Frequency",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Daytime_Sleepiness_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

## 7. Demographic factors

This section compares sleep-duration patterns across gender, age, race/ethnicity, and socioeconomic position.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def analyze_categorical(df, variable, target="Sleep_Duration_Category"):
    """
    Perform a categorical association analysis between a predictor
    and the sleep duration category.
    """

    data = df[[variable, target]].dropna()

    contingency = pd.crosstab(data[variable], data[target])

    percentages = (
        pd.crosstab(
            data[variable],
            data[target],
            normalize="index"
        ) * 100
    ).round(1)
    
    frequencies = (
    data[variable]
    .value_counts()
    .sort_index()
    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    n = contingency.values.sum()
    cramers_v = np.sqrt(
        chi2 / (n * (min(contingency.shape) - 1))
    )

    return {
    "frequencies": frequencies,
    "contingency": contingency,
    "percentages": percentages,
    "chi2": chi2,
    "p": p,
    "dof": dof,
    "cramers_v": cramers_v,
    "expected": expected
    }

In [ ]:
gender = analyze_categorical(df, "Gender")
print(gender["contingency"])

In [ ]:
print(gender["percentages"])

In [ ]:
print(f"Chi-square : {gender['chi2']:.2f}")
print(f"Degrees of freedom : {gender['dof']}")
print(f"p-value : {gender['p']:.4f}")
print(f"Cramer's V : {gender['cramers_v']:.3f}")

In [ ]:
print(df.groupby("Gender")["Average_Sleep_Hours"].describe())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

ax = sns.violinplot(
    data=df,
    x="Gender",
    y="Average_Sleep_Hours",
    palette=["#F53809", "#241DAD"],
    inner="box",      # Muestra el boxplot (mediana + cuartiles)
    cut=0,
    linewidth=1.2
)

plt.title(
    "Distribution de la sleep duration por sexo",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")
plt.ylabel("Horas medias de sueño")

plt.grid(axis="y", linestyle="--", alpha=0.3)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
bins = [20, 30, 40, 50, 60, 70, 81]
labels = ["20–29", "30–39", "40–49", "50–59", "60–69", "70–80"]

df["Age_Group_10"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels,
    right=False
)

analyze_categorical(df, "Age_Group_10")

In [ ]:
print(df["Age_Group_10"].value_counts().sort_index())
age = analyze_categorical(df, "Age_Group_10")
print(age["percentages"])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))


colors = {
    "Recommended Sleep": "#2ca02c",   # Verde
    "Short Sleep": "#ff7f0e",         # Naranja
    "Long Sleep": "#1f77b4"           # Azul
}

age["percentages"].plot(
    kind="bar",
    ax=ax,
    width=0.8,
    color=[colors[col] for col in age["percentages"].columns]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=9, padding=2)

ax.set_title("Distribution de la sleep duration según el grupo de edad", fontsize=14)
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 70)

ax.grid(axis="y", linestyle="--", alpha=0.4)

ax.legend(
    ["Sueño corto (<7 h)",
        "Sueño recomendado (7–9 h)",

        "Sueño largo (>9 h)"
    ],
    title="Categorías de sueño",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd


contingency = pd.crosstab(
    df["Age_Group"],
    df["Sleep_Duration_Category"]
)


chi2, p, dof, expected = chi2_contingency(contingency)


n = contingency.values.sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print("="*40)
print("Statistical Results")
print("="*40)
print(f"Chi-square : {chi2:.2f}")
print(f"P-value    : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

In [ ]:
race_results = analyze_categorical(df_analysis, "Race")

In [ ]:
display(race_results["contingency"].sum(axis=1))

In [ ]:
display(race_results["contingency"])

In [ ]:
display(race_results["percentages"])

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,4))

sns.heatmap(
    race_results["percentages"],
    annot=True,
    cmap="Blues",
    fmt=".1f"
)

plt.title("Sleep Duration Category by Race (%)")
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
print(f"Chi²: {race_results['chi2']:.2f}")
print(f"p-value: {race_results['p']:.4f}")
print(f"Cramer's V: {race_results['cramers_v']:.3f}")

In [ ]:
df_analysis["Poverty_Group"] = pd.qcut(
    df_analysis["Poverty_Index"],
    q=5,
    labels=[
    "Lowest",
    "Low",
    "Middle",
    "High",
    "Highest"
    ]
)

print(df_analysis["Poverty_Group"].value_counts().sort_index())

In [ ]:
poverty_results = analyze_categorical(
    df_analysis,
    "Poverty_Group"
)

print(poverty_results["contingency"])

print(poverty_results["percentages"])

In [ ]:
poverty_results["percentages"].plot(
    kind="line",
    marker="o",
    figsize=(8,5)
)

plt.title("Sleep Duration Category by Poverty Index Quintiles")
plt.xlabel("Poverty Index Quintiles")
plt.ylabel("Percentage (%)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

colors = [
    "#ff7f0e",  # Short Sleep (orange)
    "#2ca02c",  # Recommended Sleep (green)
    "#1f77b4"   # Long Sleep (blue)
]

ax = (
    poverty_results["percentages"]
    .reindex(columns=["Short Sleep", "Recommended Sleep", "Long Sleep"])
    .plot(
        kind="bar",
        stacked=True,
        figsize=(9,6),
        width=0.8,
        color=colors
    )
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        fontsize=9,
        label_type="center"
    )

plt.title("Sleep Duration Category by Poverty Index Quintiles")
plt.xlabel("Poverty Index Quintiles")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)

plt.legend(
    title="Sleep Duration Category",
    labels=[
        "Short Sleep (<7 h)",
        "Recommended Sleep (7–9 h)",
        "Long Sleep (>9 h)"
    ],
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Chi-square: {poverty_results['chi2']:.2f}")
print(f"Degrees of freedom: {poverty_results['dof']}")
print(f"P-value: {poverty_results['p']:.4f}")
print(f"Cramer's V: {poverty_results['cramers_v']:.3f}")

## 8. Conclusions

The analysis identifies statistically significant but mostly weak associations, reinforcing that sleep duration is influenced by multiple biological, social, and behavioral factors.


In [ ]:
order = ["Underweight", "Normal", "Overweight", "Obese"]

plt.figure(figsize=(8,6))

sns.pointplot(
    data=df,
    x="BMI_Category",
    y="Average_Sleep_Hours",
    order=order,
    errorbar=None,
    capsize=0.15,
    markers="o",
    linestyles="-"
)

plt.title("Average Sleep Duration Across BMI Categories")
plt.xlabel("")
plt.ylabel("Average Sleep Duration (hours)")

plt.grid(axis="y", alpha=0.3)

plt.show()

## Final interpretation

Sleep duration was associated with several demographic, health, and lifestyle characteristics. Most effect sizes were weak or very weak, so statistical significance should not be confused with practical importance. The results support a multifactorial interpretation of sleep and do not establish causality.
